# LoRA Fine-Tuning — Brain Tumor MRI (3-Class Classification)

Full LoRA pipeline for Brain Tumor 3-class classification (meningioma, glioma, pituitary) on the Cheng Brain Tumor MRI benchmark.

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if not os.path.exists("DinoV2B"):
    subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/DinoV2B.git"], check=True)

os.chdir("DinoV2B")

subprocess.run(["pip", "install", "-e", ".", "-r", "requirements.txt", "-q"], check=True)

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Configuration

Load benchmark configuration for Brain Tumor MRI and default LoRA PEFT hyperparameters.

In [ ]:
"""
Load Brain Tumor MRI dataset configuration and LoRA hyperparameters.
"""
from typing import Any
from src.utils.config import load_config

cfg: dict[str, Any] = load_config("configs/brain_tumor.yaml")
peft_cfg: dict[str, Any] = load_config("configs/peft/lora.yaml")
print("Brain Tumor Dataset Configuration:", cfg["dataset"])
print("LoRA Fine-Tuning Configuration:", peft_cfg["lora"])

## 2. Build Datasets

Construct training, validation, and test PyTorch dataloaders from `cheng-brain-tumor-mri-detection.ndjson`.

In [ ]:
"""
Construct Brain Tumor MRI dataloaders for train, val, and test splits from the NDJSON manifest.
"""
from src.data import build_brain_tumor_dataloaders, build_transforms
from src.data.brain_tumor_dataset import load_ndjson_records

train_tf = build_transforms(224, cfg["preprocessing"]["normalize_mean"], cfg["preprocessing"]["normalize_std"], augment=True)
val_tf = build_transforms(224, cfg["preprocessing"]["normalize_mean"], cfg["preprocessing"]["normalize_std"], augment=False)
train_loader, val_loader, test_loader = build_brain_tumor_dataloaders(
    cfg, "cheng-brain-tumor-mri-detection.ndjson", image_root="", train_transforms=train_tf, val_transforms=val_tf
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## 3. Linear Probing Baseline

Train a frozen DINOv2 backbone paired with a trainable 3-class classification head (`MultiClassHead(768, 3)`).

In [ ]:
"""
Train and evaluate linear probing baseline on Brain Tumor MRI dataset.
"""
from typing import Any
import torch
import torch.nn as nn
from torch.optim import AdamW
from src.evaluation import Evaluator
from src.models import count_trainable_parameters, freeze_backbone, load_backbone, MultiClassHead
from src.training.losses import MultiClassCELoss
from src.utils.logging_utils import ExperimentLogger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone, _ = load_backbone()
freeze_backbone(backbone)
head_lp = MultiClassHead(in_features=768, num_classes=3)

class LinearProbeBrainTumor(nn.Module):
    """
    Combines frozen DINOv2 backbone with multi-class classification head.
    """
    def __init__(self, b: nn.Module, h: nn.Module) -> None:
        super().__init__()
        self.b = b
        self.h = h

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Forward pass extracting CLS token representation.
        """
        outputs = self.b(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.h(cls_token)

model_lp = LinearProbeBrainTumor(backbone, head_lp).to(device)
trainable_lp, total_lp = count_trainable_parameters(model_lp)
print(f"Linear Probe Trainable: {trainable_lp:,} / {total_lp:,} ({100 * trainable_lp / total_lp:.4f}%)")

criterion_ce = MultiClassCELoss()
optimizer_lp = AdamW(head_lp.parameters(), lr=5e-3, weight_decay=1e-4)

for epoch in range(10):
    model_lp.train()
    running_loss: float = 0.0
    for batch in train_loader:
        pix, lab = batch["pixel_values"].to(device), batch["labels"].to(device)
        optimizer_lp.zero_grad()
        loss = criterion_ce(model_lp(pix), lab)
        loss.backward()
        optimizer_lp.step()
        running_loss += loss.item()
    print(f"LP Epoch [{epoch + 1:02d}/10] Loss: {running_loss / len(train_loader):.4f}")

evaluator_bt = Evaluator(task="multiclass", label_names=["meningioma", "glioma", "pituitary"])
results_lp_val = evaluator_bt.evaluate(model_lp, val_loader, device=str(device))
results_lp_test = evaluator_bt.evaluate(model_lp, test_loader, device=str(device))
print(f"Linear Probing Val Accuracy: {results_lp_val['accuracy']:.4f}, Macro-F1: {results_lp_val['macro_f1']:.4f}")
print(f"Linear Probing Test Accuracy: {results_lp_test['accuracy']:.4f}, Macro-F1: {results_lp_test['macro_f1']:.4f}")

logger = ExperimentLogger()
logger.log(
    run_id="lp_brain_tumor",
    dataset="BrainTumorMRI",
    backbone="DINOv2 ViT-B/14",
    method="LinearProbing",
    accuracy=results_lp_test["accuracy"],
    macro_f1=results_lp_test["macro_f1"],
)

## 4. LoRA Fine-Tuning (r=48)

Train DINOv2 ViT-B/14 adapted with standard LoRA (r=48) using `MultiClassCELoss`.

In [ ]:
"""
Train standard LoRA model (r=48) on Brain Tumor MRI dataset and evaluate accuracy and macro-F1.
"""
from src.models import apply_lora

backbone_lora, _ = load_backbone()
backbone_lora = apply_lora(backbone_lora, {"r": 48, "lora_alpha": 16, "target_modules": ["query", "value"]})
head_lora = MultiClassHead(in_features=768, num_classes=3)
model_lora = LinearProbeBrainTumor(backbone_lora, head_lora).to(device)

trainable_lora, total_lora = count_trainable_parameters(model_lora)
print(f"LoRA Trainable: {trainable_lora:,} / {total_lora:,} ({100 * trainable_lora / total_lora:.2f}%)")

optimizer_lora = AdamW(
    [
        {"params": [p for p in model_lora.b.parameters() if p.requires_grad], "lr": 2e-4},
        {"params": model_lora.h.parameters(), "lr": 5e-3},
    ],
    weight_decay=1e-4,
)

for epoch in range(10):
    model_lora.train()
    running_loss: float = 0.0
    for batch in train_loader:
        pix, lab = batch["pixel_values"].to(device), batch["labels"].to(device)
        optimizer_lora.zero_grad()
        loss = criterion_ce(model_lora(pix), lab)
        loss.backward()
        optimizer_lora.step()
        running_loss += loss.item()
    print(f"LoRA Epoch [{epoch + 1:02d}/10] Loss: {running_loss / len(train_loader):.4f}")

results_lora_val = evaluator_bt.evaluate(model_lora, val_loader, device=str(device))
results_lora_test = evaluator_bt.evaluate(model_lora, test_loader, device=str(device))
print(f"LoRA (r=48) Val Accuracy: {results_lora_val['accuracy']:.4f}, Macro-F1: {results_lora_val['macro_f1']:.4f}")
print(f"LoRA (r=48) Test Accuracy: {results_lora_test['accuracy']:.4f}, Macro-F1: {results_lora_test['macro_f1']:.4f}")

logger.log(
    run_id="lora_r48_brain_tumor",
    dataset="BrainTumorMRI",
    backbone="DINOv2 ViT-B/14",
    method="LoRA",
    rank=48,
    accuracy=results_lora_test["accuracy"],
    macro_f1=results_lora_test["macro_f1"],
)

## 5. Rank Sweep [4, 8, 16, 32, 48]

Run rank sweep across ranks [4, 8, 16, 32, 48] on Brain Tumor MRI to observe multi-class capacity requirements.

In [ ]:
"""
Execute LoRA rank sweep over [4, 8, 16, 32, 48] on Brain Tumor MRI.
"""
from typing import Any
import pandas as pd

RANK_SWEEP_BT: list[int] = [4, 8, 16, 32, 48]
bt_sweep_records: list[dict[str, Any]] = []

for r in RANK_SWEEP_BT:
    print(f"\n=== Running Brain Tumor Rank Sweep: r={r} ===")
    bb_r, _ = load_backbone()
    bb_r = apply_lora(bb_r, {"r": r, "lora_alpha": 16, "target_modules": ["query", "value"]})
    hd_r = MultiClassHead(in_features=768, num_classes=3)
    m_r = LinearProbeBrainTumor(bb_r, hd_r).to(device)
    tr_params, _ = count_trainable_parameters(m_r)

    opt_r = AdamW(
        [
            {"params": [p for p in m_r.b.parameters() if p.requires_grad], "lr": 2e-4},
            {"params": m_r.h.parameters(), "lr": 5e-3},
        ],
        weight_decay=1e-4,
    )

    for epoch in range(8):
        m_r.train()
        for batch in train_loader:
            pix, lab = batch["pixel_values"].to(device), batch["labels"].to(device)
            opt_r.zero_grad()
            loss = criterion_ce(m_r(pix), lab)
            loss.backward()
            opt_r.step()

    test_metrics = evaluator_bt.evaluate(m_r, test_loader, device=str(device))
    print(f"Rank r={r} Test Accuracy: {test_metrics['accuracy']:.4f}, Macro-F1: {test_metrics['macro_f1']:.4f}")
    bt_sweep_records.append(
        {"rank": r, "trainable_params": tr_params, "accuracy": test_metrics["accuracy"], "macro_f1": test_metrics["macro_f1"]}
    )
    logger.log(
        run_id=f"lora_r{r}_brain_tumor",
        dataset="BrainTumorMRI",
        backbone="DINOv2 ViT-B/14",
        method="LoRA",
        rank=r,
        accuracy=test_metrics["accuracy"],
        macro_f1=test_metrics["macro_f1"],
    )

bt_df = pd.DataFrame(bt_sweep_records)
print(bt_df.to_string(index=False))

## 6. Results and Confusion Matrix

Plot the test set confusion matrix and output a detailed classification report with precision, recall, and F1 scores per tumor class.

In [ ]:
"""
Generate confusion matrix heatmap and classification report table on held-out test split.
"""
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

model_lora.eval()
all_preds: list[int] = []
all_targets: list[int] = []

with torch.no_grad():
    for batch in test_loader:
        pix = batch["pixel_values"].to(device)
        labs = batch["labels"]
        logits = model_lora(pix)
        preds = torch.argmax(logits, dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_targets.extend(labs.tolist())

class_names: list[str] = ["meningioma", "glioma", "pituitary"]
cm = confusion_matrix(all_targets, all_preds)
print("Classification Report:\n", classification_report(all_targets, all_preds, target_names=class_names))

sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title("Brain Tumor LoRA Test Confusion Matrix", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Tumor Class", fontsize=11)
ax.set_ylabel("Ground Truth Class", fontsize=11)
plt.tight_layout()
plt.show()

## 7. Log All Experiments

Review the updated centralized experiment log containing all Brain Tumor MRI runs.

In [ ]:
"""
Display full experiment logger dataframe to verify Brain Tumor MRI records.
"""
log_df = logger.to_dataframe()
print(log_df[log_df["dataset"] == "BrainTumorMRI"][["run_id", "method", "accuracy", "macro_f1"]])